# Going Classy · Object-Oriented Programming for the Training Loop

**DCA0305 · Machine Learning Based Systems Design**

In `lesson3a` the pipeline was built from loose functions and global variables (`model`, `optimizer`, `train_loader`, `losses`, ...). It works, but it does not scale as software. State is scattered, functions depend on globals, and reusing the pipeline on a new problem means copy-pasting cells.

**Going classy** means packing state and behavior together into a class. Everything the training process *is* (model, loss, optimizer, loaders, histories, device) becomes an **attribute**. Everything the training process *does* (train, validate, checkpoint, predict, plot) becomes a **method**. The result is a small, reusable training engine, a miniature version of what libraries like PyTorch Lightning, Keras, and fastai provide.

## 🎯 Learning objectives

1. Map each global variable and function from `lesson3a` to its new home inside a class.
2. Explain the roles of `__init__`, `self`, public methods, and underscore-prefixed methods.
3. Explain why the step-function factories no longer need arguments.
4. Use the class to train, checkpoint, resume, and predict in a handful of lines.
5. Extend the class with new behavior (exercises).

## 🧠 Retention protocol

Same rules as before. 🔮 *Predict* prompts ask you to commit to an answer before running a cell, ✅ *Check yourself* boxes hide their answers until you have tried to retrieve them, and the 📝 final self-test is meant for a day or two after class.


# 0. Warm-up · The five OOP ideas we need

We only need a handful of Python OOP concepts, so here is a compact, self-contained refresher.

1. **Class vs instance.** A class is the blueprint (`Architecture`), an instance is one concrete object built from it (`arch = Architecture(...)`). Every instance carries its own independent state, so two experiments can coexist without clobbering each other's losses.
2. **`__init__`** runs once, at construction time. Its job is to receive the ingredients and initialize every attribute the object will ever use. Initializing all attributes in `__init__`, even the ones that start as `None`, documents the full state of the object in one place.
3. **`self`** is the instance itself, passed automatically as the first argument of every method. `self.model` means "this object's model". This is how methods share state without global variables.
4. **Naming convention for privacy.** Python has no `private` keyword. By convention, a leading underscore (`_make_train_step_fn`, `_mini_batch`) signals "internal machinery, not part of the public interface". Users of the class should only need `to`, `set_loaders`, `train`, `save_checkpoint`, `load_checkpoint`, `predict`, and `plot_losses`.
5. **Methods returning functions still work.** A closure created inside a method can capture `self`. That is why our step-function factories lose all their arguments in this version. They read `self.model`, `self.loss_fn`, and `self.optimizer` directly.


In [ ]:
# A 60-second demo of the ideas above, before the real class
class Counter:
    def __init__(self, start=0):
        self.value = start          # attribute = state

    def increment(self):            # public method = behavior
        self.value += 1
        return self.value

    def _reset(self):               # underscore = internal use
        self.value = 0

a = Counter()
b = Counter(100)
a.increment(); a.increment()
b.increment()
print(a.value, b.value)   # independent state per instance -> 2 and 101

✅ **Check yourself**

<details><summary>1. When exactly does <code>__init__</code> run, and how many times per object?</summary>

Exactly once per instance, at the moment you write `Architecture(model, loss_fn, optimizer)`. It never runs again for that object.
</details>

<details><summary>2. Why does <code>a.increment()</code> not need any argument even though the method signature is <code>increment(self)</code>?</summary>

Python fills in `self` automatically with the instance the method was called on. `a.increment()` is equivalent to `Counter.increment(a)`.
</details>

<details><summary>3. What does the leading underscore in <code>_reset</code> actually enforce?</summary>

Nothing at the language level. It is a convention that tells other developers "this is internal, do not rely on it". Python trusts programmers instead of enforcing access control.
</details>


# 1. Imports


In [ ]:
import numpy as np
import datetime
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, TensorDataset, DataLoader
from torch.utils.data.dataset import random_split
import torch.optim as optim
import torch.nn as nn
%matplotlib inline
plt.style.use('fivethirtyeight')

print(f"PyTorch version: {torch.__version__}")

# 2. The `Architecture` class

## 2.1 The map from `lesson3a` to here

Keep this translation table in mind while reading the class. Nothing conceptually new happens, the pieces just move into an object.

| lesson3a (functions + globals) | lesson3b (class) |
|---|---|
| globals `model`, `loss_fn`, `optimizer`, `device` | attributes set in `__init__` |
| globals `train_loader`, `val_loader` | attributes set by `set_loaders()` |
| globals `losses`, `val_losses`, epoch counter | attributes `self.losses`, `self.val_losses`, `self.total_epochs` |
| `make_train_step_fn(model, loss_fn, optimizer)` | `self._make_train_step_fn()` with **no args** |
| `make_val_step_fn(model, loss_fn)` | `self._make_val_step_fn()` with **no args** |
| `mini_batch(device, loader, step_fn)` | `self._mini_batch(validation=...)` |
| the raw training loop | `self.train(n_epochs)` |
| save / load cells | `save_checkpoint()` / `load_checkpoint()` |
| prediction cell | `predict(x)` |

## 2.2 Design decisions worth noticing

- **Constructor takes only the essentials.** Model, loss, and optimizer define *what* is trained. Loaders define *which data*, so they arrive later through `set_loaders`. This separation lets you reuse one configured `Architecture` with different datasets.
- **Deferred attributes start as `None`.** `train_loader` and `val_loader` are declared in `__init__` even though they are unknown at that point. All state is visible in one place.
- **The argument-less factories.** `_make_train_step_fn` builds a closure that captures `self`, so the returned function reads `self.model`, `self.loss_fn`, `self.optimizer` live. If you later swap `self.optimizer`, the step function sees the new one automatically.
- **`_mini_batch(validation=...)`** merges the two nearly identical loops from `lesson3a` into one, selecting loader and step function by a boolean.
- **`total_epochs` is cumulative.** Calling `train` twice for 200 and 50 epochs leaves `total_epochs == 250`, and the loss histories concatenate. This is what makes *resuming* seamless.
- **`set_seed` goes beyond `manual_seed`.** The two cuDNN flags trade a bit of GPU speed for determinism of convolution algorithms. Irrelevant for a linear model, essential to know for CNNs.


In [ ]:
class Architecture:
    """A minimal, reusable training engine for PyTorch models.

    Packs together the model, loss function, optimizer, data loaders,
    loss histories, and device handling. Public interface:
    to, set_loaders, set_seed, train, save_checkpoint, load_checkpoint,
    predict, plot_losses.
    """

    def __init__(self, model, loss_fn, optimizer):
        # We start by storing the constructor arguments as attributes,
        # so every method can reach them through `self`
        self.model = model
        self.loss_fn = loss_fn
        self.optimizer = optimizer
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        # Send the model to the chosen device right away (the model LIVES there)
        self.model.to(self.device)

        # These attributes are defined here, but since they are
        # not informed at construction time, we keep them as None
        self.train_loader = None
        self.val_loader = None

        # These attributes are computed internally during training
        self.losses = []
        self.val_losses = []
        self.total_epochs = 0

        # Creates the train/val step functions.
        # Note: NO ARGS! The closures capture `self` and use the
        # class attributes directly.
        self.train_step_fn = self._make_train_step_fn()
        self.val_step_fn = self._make_val_step_fn()

    def to(self, device):
        """Move the engine (model + future batches) to another device."""
        # Sets the corresponding attribute (used later by the mini-batch
        # loop) and sends the model to the device
        try:
            self.device = device
            self.model.to(self.device)
        except RuntimeError:
            self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
            print(f"Couldn't send it to {device}, sending it to {self.device} instead.")
            self.model.to(self.device)

    def set_loaders(self, train_loader, val_loader=None):
        """Attach the data loaders (validation is optional)."""
        self.train_loader = train_loader
        self.val_loader = val_loader

    def _make_train_step_fn(self):
        # This method needs no args. The closure below refers to
        # self.model, self.loss_fn and self.optimizer directly.
        def perform_train_step_fn(x, y):
            # Sets model to TRAIN mode
            self.model.train()

            # Step 1 - Computes model's predictions - forward pass
            yhat = self.model(x)
            # Step 2 - Computes the loss
            loss = self.loss_fn(yhat, y)
            # Step 3 - Computes gradients for all parameters
            loss.backward()
            # Step 4 - Updates parameters, then clears gradients
            # (PyTorch accumulates gradients by default)
            self.optimizer.step()
            self.optimizer.zero_grad()

            # Returns a plain float, detached from the graph
            return loss.item()

        return perform_train_step_fn

    def _make_val_step_fn(self):
        def perform_val_step_fn(x, y):
            # Sets model to EVAL mode
            self.model.eval()

            # Step 1 - forward pass
            yhat = self.model(x)
            # Step 2 - loss
            loss = self.loss_fn(yhat, y)
            # No Steps 3 and 4. We never update parameters during evaluation.
            return loss.item()

        return perform_val_step_fn

    def _mini_batch(self, validation=False):
        # One loop serves both regimes. The `validation` flag selects
        # which loader and which step function to use.
        if validation:
            data_loader = self.val_loader
            step_fn = self.val_step_fn
        else:
            data_loader = self.train_loader
            step_fn = self.train_step_fn

        if data_loader is None:
            return None

        # Once loader and step function are chosen, this is the same
        # mini-batch loop we had before
        mini_batch_losses = []
        for x_batch, y_batch in data_loader:
            # Data lives on the CPU; each batch visits the device only here
            x_batch = x_batch.to(self.device)
            y_batch = y_batch.to(self.device)

            mini_batch_loss = step_fn(x_batch, y_batch)
            mini_batch_losses.append(mini_batch_loss)

        loss = np.mean(mini_batch_losses)
        return loss

    def set_seed(self, seed=42):
        """Make the training run reproducible (RNGs + cuDNN determinism)."""
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.manual_seed(seed)
        np.random.seed(seed)

    def train(self, n_epochs, seed=42):
        """Run the full training loop for n_epochs (cumulative)."""
        # To ensure reproducibility of the training process
        self.set_seed(seed)

        for epoch in range(n_epochs):
            # Keeps a CUMULATIVE count of epochs across multiple calls
            self.total_epochs += 1

            # inner loop - training with mini-batches (gradients ON)
            loss = self._mini_batch(validation=False)
            self.losses.append(loss)

            # VALIDATION - no gradients!
            with torch.no_grad():
                val_loss = self._mini_batch(validation=True)
                self.val_losses.append(val_loss)

    def save_checkpoint(self, filename):
        """Persist everything needed to resume training later."""
        checkpoint = {'epoch': self.total_epochs,
                      'model_state_dict': self.model.state_dict(),
                      'optimizer_state_dict': self.optimizer.state_dict(),
                      'loss': self.losses,
                      'val_loss': self.val_losses}

        torch.save(checkpoint, filename)

    def load_checkpoint(self, filename):
        """Restore model, optimizer and histories from a checkpoint file."""
        # map_location makes a GPU-saved checkpoint load on CPU-only machines
        checkpoint = torch.load(filename,
                                weights_only=False,
                                map_location=self.device)

        # Restore state for model and optimizer
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

        self.total_epochs = checkpoint['epoch']
        self.losses = checkpoint['loss']
        self.val_losses = checkpoint['val_loss']

        self.model.train()   # always use TRAIN mode when resuming training

    def predict(self, x):
        """Run inference on new data (NumPy in, NumPy out)."""
        # Set the model to evaluation mode for predictions
        self.model.eval()
        # Takes a NumPy input and makes it a float tensor
        x_tensor = torch.as_tensor(x).float()
        # IMPROVEMENT: no computation graph during inference
        with torch.no_grad():
            y_hat_tensor = self.model(x_tensor.to(self.device))
        # Set it back to train mode
        self.model.train()
        # Brings the result to CPU and back to NumPy
        return y_hat_tensor.cpu().numpy()

    def plot_losses(self):
        """Plot cumulative training and validation loss histories."""
        fig = plt.figure(figsize=(10, 4))
        plt.plot(self.losses, label='Training Loss', c='b')
        plt.plot(self.val_losses, label='Validation Loss', c='r')
        plt.yscale('log')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()
        plt.tight_layout()
        return fig

### 🔮 Predict

Without running anything, answer. If two instances are created, `arch1 = Architecture(m1, loss, opt1)` and `arch2 = Architecture(m2, loss, opt2)`, and only `arch1.train(100)` is called, what is `arch2.total_epochs`? Why?

<details><summary>Answer</summary>

`arch2.total_epochs` is still `0`. Attributes are per-instance state. Each object created by the class carries its own `losses`, `val_losses`, and `total_epochs`, so training one instance never touches another. This isolation is precisely what the global-variable version could not offer.
</details>

✅ **Check yourself**

<details><summary>1. Why can <code>_make_train_step_fn</code> drop all three arguments that its lesson3a counterpart required?</summary>

Because the inner function is a closure over `self`. It reads `self.model`, `self.loss_fn`, and `self.optimizer` at call time, so the ingredients no longer travel as arguments. This also means the step function always sees the *current* attributes, even if they are replaced later.
</details>

<details><summary>2. What does <code>_mini_batch</code> return if <code>set_loaders</code> was never called, and why is that a reasonable design?</summary>

It returns `None`, because both loaders are still `None`. This makes the validation loader genuinely optional. You can train without validation and `val_losses` simply fills with `None` instead of crashing.
</details>

<details><summary>3. Why is <code>total_epochs</code> incremented inside the loop instead of doing <code>self.total_epochs += n_epochs</code> once?</summary>

Functionally similar here, but incrementing per epoch keeps the counter correct even if training is interrupted mid-way (exception, keyboard interrupt), and it gives future extensions (loggers, schedulers, early stopping) an accurate epoch number at every step.
</details>

<details><summary>4. In <code>predict</code>, why convert back with <code>.cpu().numpy()</code>?</summary>

NumPy only understands CPU memory. If the model ran on the GPU, the output tensor lives there, and `.numpy()` would fail. `.cpu()` copies it back first. With `no_grad`, `.detach()` is no longer needed since no graph was recorded.
</details>


# 3. Put it all together

Watch how short the user-facing code becomes. Data generation and preparation are unchanged from `lesson3a`. Model configuration no longer mentions the device (the class handles it). Training is three lines.

## 3.1 Data Generation


In [ ]:
true_b = 1
true_w = 2
N = 100

# Data Generation
np.random.seed(42)
x = np.random.rand(N, 1)
y = true_b + true_w * x + (.1 * np.random.randn(N, 1))

## 3.2 Data Preparation

Same recipe as before. Tensors are built before the split, the split gets its own generator for reproducibility, and only the train loader shuffles.


In [ ]:
torch.manual_seed(13)

# Builds tensors from numpy arrays BEFORE split (CPU tensors, on purpose)
x_tensor = torch.as_tensor(x).float()
y_tensor = torch.as_tensor(y).float()

# Builds dataset containing ALL data points
dataset = TensorDataset(x_tensor, y_tensor)

# Performs the split
ratio = .8
n_total = len(dataset)
n_train = int(n_total * ratio)
n_val = n_total - n_train

split_generator = torch.Generator().manual_seed(42)
train_data, val_data = random_split(dataset, [n_train, n_val], generator=split_generator)

# Builds a loader for each set
train_loader = DataLoader(dataset=train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(dataset=val_data, batch_size=16)

## 3.3 Model Configuration

Note what is *missing* compared to `lesson3a`. No `device` line and no `.to(device)`. The `Architecture` constructor takes care of device placement. Configuration now describes only the learning problem (model, loss, optimizer, learning rate).


In [ ]:
# Sets learning rate - this is "eta" ~ the "n"-like Greek letter
lr = 0.1

torch.manual_seed(42)
# Now we can create a model (no .to(device) needed here anymore!)
model = nn.Sequential(nn.Linear(1, 1))

# Defines an SGD optimizer to update the parameters
optimizer = optim.SGD(model.parameters(), lr=lr)

# Defines an MSE loss function
loss_fn = nn.MSELoss(reduction='mean')

## 3.4 Training in three lines


In [ ]:
n_epochs = 200

arch = Architecture(model, loss_fn, optimizer)
arch.set_loaders(train_loader, val_loader)
arch.train(n_epochs=n_epochs)

print(f"total epochs so far: {arch.total_epochs}")
print(model.state_dict())

In [ ]:
fig = arch.plot_losses()
plt.show()

Both curves settle near the irreducible noise floor ($0.1^2 = 0.01$), exactly as in `lesson3a`. Same math, better software.

# 4. Checkpointing and Resuming Training

## 4.1 Save


In [ ]:
# checkpointing
arch.save_checkpoint('model_checkpoint.pth')
print("saved.")

## 4.2 Resume in a "fresh session"

The cell below simulates starting over (imagine you closed Colab and came back tomorrow). We rebuild model, optimizer, and loss from scratch, wrap them in a **new** `Architecture`, and restore the checkpoint into it.

### 🔮 Predict

Immediately after `load_checkpoint`, before any new training, what will `print(model.state_dict())` show? Fresh random weights or the trained weights (w ≈ 2, b ≈ 1)? Commit to an answer, then run.


In [ ]:
# Resuming Training

# Sets learning rate
lr = 0.1

torch.manual_seed(42)
# A brand-new model with fresh random weights...
model = nn.Sequential(nn.Linear(1, 1))

optimizer = optim.SGD(model.parameters(), lr=lr)
loss_fn = nn.MSELoss(reduction='mean')

# ...restored from the checkpoint by a NEW engine instance
new_arch = Architecture(model, loss_fn, optimizer)
new_arch.load_checkpoint('model_checkpoint.pth')

print(f"restored at epoch {new_arch.total_epochs}")
print(model.state_dict())   # trained weights, not random ones

The state dict shows the *trained* parameters. `load_state_dict` copies the checkpoint tensors into the freshly created model in place. The random initialization was simply overwritten.

Now we keep training for 50 more epochs. Watch two things. `total_epochs` continues from 200 to 250, and the loss plot shows the whole 250-epoch history because the loss lists were restored too.


In [ ]:
new_arch.set_loaders(train_loader, val_loader)
new_arch.train(n_epochs=50)

print(f"total epochs now: {new_arch.total_epochs}")

In [ ]:
fig = new_arch.plot_losses()
plt.show()

The curve continues flat after epoch 200, which is expected. The model had already converged, so 50 extra epochs just bounce around the noise floor. In a real project this is exactly how you would *verify* that resuming worked, no discontinuity or spike at the resume point.

✅ **Check yourself**

<details><summary>1. Why do we rebuild <code>model</code> and <code>optimizer</code> before calling <code>load_checkpoint</code> instead of loading into thin air?</summary>

`load_state_dict` needs existing objects with matching architecture to copy tensors into. The checkpoint stores *values*, not the objects themselves. That is also why the model class/structure must match the one used at save time.
</details>

<details><summary>2. What would a spike in the loss right after epoch 200 indicate?</summary>

That some training state was not restored properly. The classic cause with adaptive optimizers (Adam) is a lost optimizer state. With plain SGD, a likely cause would be loading weights but a wrong learning rate or corrupted loaders.
</details>


# 5. Make Predictions

`predict` accepts NumPy and returns NumPy, hiding all tensor and device juggling. Note the input shape. `reshape(-1, 1)` makes it `(3, 1)`, three samples of one feature each, matching what `nn.Linear(1, 1)` expects.


In [ ]:
new_data = np.array([.5, .3, .7]).reshape(-1, 1)
print(new_data, new_data.shape)

In [ ]:
predictions = new_arch.predict(new_data)
print(predictions)
# sanity check against the noise-free generating line y = 1 + 2x
print("expected:", (true_b + true_w * new_data).ravel())

Predictions land within a couple of hundredths of the true line, the same small gap we diagnosed in `lesson3a` (the model fits the noisy sample, not the platonic line).

# 6. 📝 Final self-test (spaced retrieval)

<details><summary>1. Reconstruct the public interface of <code>Architecture</code> from memory and say in one sentence what each method does.</summary>

`to(device)` moves the engine to a device. `set_loaders(train, val)` attaches data. `set_seed(seed)` pins RNGs. `train(n_epochs)` runs the cumulative training loop. `save_checkpoint(f)` / `load_checkpoint(f)` persist and restore full training state. `predict(x)` runs no-grad inference, NumPy in and NumPy out. `plot_losses()` plots both histories.
</details>

<details><summary>2. Which three pieces of lesson3a state became constructor arguments, and which two arrive later by a setter? Why the split?</summary>

Model, loss, and optimizer come in the constructor because they define the learning problem. Loaders come through `set_loaders` because data is a separate concern. The same engine can be reused with different datasets.
</details>

<details><summary>3. Name the two bug-class improvements made to the original class in this notebook and the failure each prevents.</summary>

`torch.no_grad()` inside `predict` (prevents useless graph construction and memory growth during inference) and `map_location` in `load_checkpoint` (prevents a `RuntimeError` when loading GPU checkpoints on CPU machines).
</details>

<details><summary>4. Your colleague calls <code>arch.train(100)</code> and gets <code>losses == [None]*100</code>. What did they forget?</summary>

`set_loaders`. With `train_loader is None`, `_mini_batch` returns `None` by design, so the loop "runs" without training.
</details>

<details><summary>5. In what sense is <code>Architecture</code> a tiny PyTorch Lightning?</summary>

Both separate the *science* (model, loss, optimizer, data) from the *engineering* (device placement, loops, checkpointing, logging). Lightning generalizes exactly this pattern with hooks, callbacks, multi-GPU support, and logging integrations.
</details>

# 7. 🏋️ Exercises

Extend the class. That is the whole point of having one.

1. **`__repr__`.** Implement it so `print(arch)` shows the device, total epochs, and number of trainable parameters (hint, `p.numel() for p in self.model.parameters() if p.requires_grad`).
2. **Progress logging.** Add a `verbose` argument to `train` that prints train/val losses every k epochs.
3. **Early stopping.** Add `train(..., patience=None)`. When `patience` is an int, stop if validation loss has not improved for that many epochs, and keep the best `state_dict` in memory.
4. **Learning-rate scheduler.** Accept an optional `torch.optim.lr_scheduler` in the constructor and call `scheduler.step()` once per epoch inside `train`.
5. **TensorBoard.** Add a `set_tensorboard(name)` method that creates a `SummaryWriter` and logs both losses per epoch (this is where the `datetime` import becomes useful for run names).
6. **Robustness.** `predict` currently crashes on a Python list input of shape `(3,)`. Make it accept lists and 1-D arrays by reshaping internally, and document the accepted shapes in the docstring.
7. **Challenge.** Reuse `Architecture` *unchanged* on a different problem, e.g. a quadratic dataset with `nn.Sequential(nn.Linear(1, 8), nn.ReLU(), nn.Linear(8, 1))`. Nothing in the class should need editing. That is the definition of a good abstraction.

# 📚 References

- Daniel Voigt Godoy, *Deep Learning with PyTorch Step-by-Step*, Chapter 2.1 (Going Classy).
- PyTorch docs. *Saving and Loading Models*, `torch.optim.lr_scheduler`, `torch.utils.tensorboard`.
